In [2]:
import re
import spacy
from spacy.matcher import Matcher

nlp = spacy.load("en_core_web_sm")

In [3]:
class InformationExtractor:
    def __init__(self):
        self.matcher = Matcher(nlp.vocab)
        self._setup_custom_rules()

    def _setup_custom_rules(self):
        """Define rule-based token patterns for academic/administrative actions."""
        action_pattern = [
            {"LOWER": {"IN": ["must", "required", "requested", "should"]}},
            {"OP": "?"}, # Optional adverb/particle
            {"POS": "VERB"}
        ]
        self.matcher.add("ACTION_ITEM", [action_pattern])

    def extract_entities(self, text: str) -> dict:
        doc = nlp(text)
        
        emails = re.findall(r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}', text)
        urls = re.findall(r'https?://[^\s]+', text)
        
        dates_and_times = []
        organizations = []
        persons = []
        
        for ent in doc.ents:
            if ent.label_ in ["DATE", "TIME"]:
                dates_and_times.append(ent.text)
            elif ent.label_ == "ORG":
                organizations.append(ent.text)
            elif ent.label_ == "PERSON":
                persons.append(ent.text)

        matches = self.matcher(doc)
        extracted_actions = []
        for match_id, start, end in matches:
            span = doc[start:end]
            extracted_actions.append(span.sent.text.strip())

        return {
            "deadlines_and_dates": list(set(dates_and_times)),
            "contact_emails": list(set(emails)),
            "action_required": list(set(extracted_actions)),
            "organizations": list(set(organizations)),
            "contacts_persons": list(set(persons)),
            "urls": list(set(urls))
        }

# Instantiate Extractor
extractor = InformationExtractor()
print("NER Engine initialized successfully!")

NER Engine initialized successfully!


In [4]:
sample_notice_text = """
OFFICE OF THE ACADEMIC CONTROLLER
Date: September 24, 2026

Circular regarding End-Semester Project Submissions:
All final-year students are hereby notified that they must submit their capstone proposal PDFs on the student portal. 
The absolute deadline for submission is October 15, 2026. Late submissions will not be entertained by the committee.

Course instructors are requested to verify project group allocations by October 18, 2026.
For any technical issues, please email support@university.edu or visit https://portal.university.edu.
"""

In [5]:
extracted_metadata=extractor.extract_entities(sample_notice_text)

import json 
print(json.dumps(extracted_metadata, indent=2))

{
  "deadlines_and_dates": [
    "October 15, 2026",
    "September 24, 2026",
    "October 18, 2026"
  ],
  "contact_emails": [
    "support@university.edu"
  ],
  "action_required": [
    "Course instructors are requested to verify project group allocations by October 18, 2026.",
    "OFFICE OF THE ACADEMIC CONTROLLER\nDate: September 24, 2026\n\nCircular regarding End-Semester Project Submissions:\nAll final-year students are hereby notified that they must submit their capstone proposal PDFs on the student portal."
  ],
  "organizations": [
    "End-Semester Project Submissions"
  ],
  "contacts_persons": [],
  "urls": [
    "https://portal.university.edu."
  ]
}


In [6]:
from transformers import AutoTokenizer, AutoModelForTokenClassification, Trainer, TrainingArguments

label_list = ["O", "B-DEADLINE", "I-DEADLINE", "B-ACTION", "I-ACTION", "B-CONTACT", "I-CONTACT"]
label2id = {label: i for i, label in enumerate(label_list)}
id2label = {i: label for i, label in enumerate(label_list)}

In [7]:
model_id = "dslim/bert-base-NER" # Pretrained token classification base
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForTokenClassification.from_pretrained(
    model_id, 
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True # Replaces classification head for custom classes
)

print("Done")

c:\Users\a_sri\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\a_sri\.cache\huggingface\hub\models--dslim--bert-base-NER. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 921.93it/s, Materializing param=classi

Done


In [ ]:
from datasets import load_dataset

# Load official announcement/email summarization dataset
aeslc = load_dataset("aeslc")

# View a sample administrative record
print("Raw Notice/Email Body:\n", aeslc["train"][0]["email_body"])
print("\nTarget Summary:\n", aeslc["train"][0]["subject_line"])